# Q1 — o tamanho da diferença

Objetivo: investigar a diferença nas notas da prova de Ciências da Natureza (CN) do ENEM entre as redes de ensino, as faixas de renda per capita e os níveis de escolaridade dos pais. Essa diferença muda ao longo do tempo?

- **Por renda e escolaridade dos pais:** só de 2020 a 2023 (`fct_individual`), porque em 2024 e 2025 a nota não se liga mais ao questionário socioeconômico, como explicado na D002.

- **Por rede:** de 2020 a 2025 (`agg_rede_ano`), com atenção à quebra de série entre 2023 e 2024, também explicada na D002. Essa parte já foi analisada em `01_q1_tamanho_da_diferenca.ipynb`.

## Configurações iniciais

In [ ]:
from src.db import conectar
from src import graficos as g

con = conectar(read_only=True)
g.aplicar_tema()

## Considerações iniciais

A partir de 2024, os microdados do ENEM passaram a ser divididos em duas tabelas: `PARTICIPANTES` e `RESULTADOS`. A primeira traz os dados de perfil e do questionário socioeconômico. A segunda traz os dados de escola, presença, notas, respostas e gabaritos.

O próprio dicionário de 2024 afirma que não é possível relacionar as duas tabelas, porque elas não têm chave em comum. Portanto, não é possível relacionar a nota de um participante com o seu perfil socioeconômico.

Por isso, toda a análise deste notebook se limita ao período de **2020 a 2023**.

Para mais detalhes, consulte a D002 (`../docs/decisoes.md`).

## Análise da diferença por renda

Vamos selecionar as colunas de interesse da `fct_individual`:

In [ ]:
renda = con.sql("""
    SELECT ano, faixa_renda_per_capita,
        COUNT(*) AS qtd_com_nota,
        AVG(nota_cn) AS media,
        MEDIAN(nota_cn) AS mediana,
        STDDEV_SAMP(nota_cn) AS desvio_padrao
    FROM fct_individual
    WHERE tem_nota_cn AND tem_questionario
    GROUP BY ALL
    ORDER BY ano, faixa_renda_per_capita
""").df()
print(renda.round(1).to_string())

A coluna `faixa_renda_per_capita` foi escolhida como referência para as análises de renda. Ela é medida em salários mínimos (SM) e dividida em quatro faixas (até 1/2 SM, de 1/2 a 1 SM, de 1 a 2 SM e acima de 2 SM). Veja os detalhes na D006 (`../docs/decisoes.md`).

Para montar o gráfico de comparação, vamos usar:

In [ ]:
ordem_renda = ["ate_meio_sm", "meio_a_1_sm", "1_a_2_sm", "acima_2_sm"]
rotulos_renda = {"ate_meio_sm": "Até 1/2 SM",
                "meio_a_1_sm": "De 1/2 a 1 SM",
                "1_a_2_sm": "De 1 a 2 SM",
                "acima_2_sm": "Acima de 2 SM"}

In [ ]:
fig, ax = g.serie_temporal(
    renda, x="ano", y="media", grupo="faixa_renda_per_capita",
    ordem=ordem_renda, rotulos=rotulos_renda, sequencial=True,
    legenda=False,
    titulo="Média em Ciências da Natureza por renda per capita — MG, 2020–2023",
    subtitulo="Concluintes com nota na prova e questionário respondido",
    rotulo_y="Nota média (pontos)",
    nota="Fonte: microdados do ENEM (INEP). SM: salário mínimo do ano. "
        "Renda per capita estimada pelo ponto médio da faixa (D006).",
    figsize=(8, 5),
)
g.salvar(fig, "q1_media_cn_por_renda", dpi=300)

O gráfico mostra uma diferença clara e regular entre as faixas de renda. No entanto, é preciso considerar que, provavelmente, a maioria dos alunos de renda alta estuda nas redes privada e federal, e os alunos de renda mais baixa estudam, com frequência, na rede estadual/municipal. Então, parte da diferença por renda pode ser a diferença por rede, e vice-versa.

### A distância entre as faixas extremas, ano a ano

Vamos analisar, ano a ano, como a média em Ciências da Natureza varia entre as faixas. Na tabela a seguir, a coluna `extremos` mostra a diferença entre a média da faixa de renda per capita mais alta e a da mais baixa, e as colunas `passo_1` a `passo_3` mostram a diferença entre cada faixa e a faixa imediatamente abaixo.

In [ ]:
piv = renda.pivot(index="ano", columns="faixa_renda_per_capita", values="media")[ordem_renda]
gap_renda = piv.assign(
    extremos=piv["acima_2_sm"] - piv["ate_meio_sm"],
    passo_1=piv["meio_a_1_sm"] - piv["ate_meio_sm"],
    passo_2=piv["1_a_2_sm"] - piv["meio_a_1_sm"],
    passo_3=piv["acima_2_sm"] - piv["1_a_2_sm"],
).round(1)
print(gap_renda.to_string())

Também podemos medir a diferença entre as faixas extremas numa escala comum, usando o **d de Cohen**:

In [ ]:
ext = renda[renda["faixa_renda_per_capita"].isin(["ate_meio_sm", "acima_2_sm"])]
ext = ext.pivot(index="ano", columns="faixa_renda_per_capita",
                values=["media", "desvio_padrao", "qtd_com_nota"])
n1, n2 = ext[("qtd_com_nota", "ate_meio_sm")], ext[("qtd_com_nota", "acima_2_sm")]
s1, s2 = ext[("desvio_padrao", "ate_meio_sm")], ext[("desvio_padrao", "acima_2_sm")]
dp_comb = (((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2)) ** 0.5
d = (ext[("media", "acima_2_sm")] - ext[("media", "ate_meio_sm")]) / dp_comb
print(d.round(2).to_string())

Os valores do *d* de Cohen entre as faixas extremas, ano a ano, estão na tabela:

| Ano | *d* de Cohen |
|---|---|
| 2020 | 1,47 |
| 2021 | 1,39 |
| 2022 | 1,24 |
| 2023 | 1,38 |

O *d* de Cohen entre as faixas extremas fica entre 1,24 e 1,47, bem acima de 0,8, o valor de referência para um efeito "grande". Um *d* em torno de 1,4 indica que um aluno típico da faixa acima de 2 SM tem nota maior que a de cerca de 9 em cada 10 alunos da faixa até 1/2 SM. Em outras palavras, cerca de 92% dos alunos da faixa de renda mais baixa têm nota abaixo da média da faixa mais alta.

Os passos entre faixas vizinhas ficam entre 25 e 38 pontos, em torno de 30 na maioria dos anos. A média sobe de forma regular de uma faixa para a outra, sem um salto concentrado num ponto. Como os limites das faixas dobram de uma faixa para a seguinte (0,5, 1 e 2 SM, pela definição da D006), e isso corresponde a passos parecidos na nota, pode ser que a nota cresça com o logaritmo da renda. Essa hipótese será explorada na Q3.

O ano de 2022 tem o menor gap entre os extremos (80,2 pontos) e o menor *d* (1,24). Se a prova de 2022 tivesse apenas encolhido a escala das notas, o gap em pontos cairia, mas o *d* ficaria parecido, porque o desvio-padrão encolheria junto. Como o *d* também cai, a diferença entre os grupos em 2022 diminuiu mais do que a dispersão geral. Esse efeito também pode ser observado no gráfico.

O *d* de Cohen também permite comparar a renda com a rede numa escala comum:

In [ ]:
r = con.sql("""
    SELECT ano, rede, media_nota_cn AS media,
        desvio_padrao_nota_cn AS dp, qtd_com_nota_cn AS n
    FROM agg_rede_ano
    WHERE periodo = '2020-2023' AND rede IN ('privada', 'estadual_municipal')
""").df().pivot(index="ano", columns="rede")
n1, n2 = r[("n", "estadual_municipal")], r[("n", "privada")]
s1, s2 = r[("dp", "estadual_municipal")], r[("dp", "privada")]
dp_comb = (((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2)) ** 0.5
d_rede = (r[("media", "privada")] - r[("media", "estadual_municipal")]) / dp_comb
print(d_rede.round(2).to_string())

Com isso, podemos comparar a diferença por renda com a diferença por rede:

| Ano | *d* por renda (extremos) | *d* por rede (privada × estadual/municipal) |
|----|------------------------|----------------------------|
| 2020 | 1,47 | 1,18 |
| 2021 | 1,39 | 1,16 |
| 2022 | 1,24 | 1,01 |
| 2023 | 1,38 | 1,09 |

Tanto a diferença por renda quanto a diferença por rede são grandes. A diferença por renda é maior que a diferença por rede em todos os anos, por uma margem de 0,2 a 0,3. E, mais uma vez, 2022 tem o menor valor nas duas colunas. Apesar de esse resultado chamar a atenção, ele tem uma limitação.

O *d* da rede vem da `agg_rede_ano` (concluintes com escola mineira informada). Já o *d* da renda vem da `fct_individual` (todos os concluintes com questionário, com ou sem escola informada). Para comparar na mesma população, precisamos recalcular o *d* da rede na `fct_individual`, filtrando quem tem escola:

In [ ]:
r2 = con.sql("""
    SELECT ano, rede,
        AVG(nota_cn) AS media, STDDEV_SAMP(nota_cn) AS dp, COUNT(*) AS n
    FROM fct_individual
    WHERE tem_nota_cn AND tem_escola
    AND rede IN ('privada', 'estadual_municipal')
    GROUP BY ALL
""").df().pivot(index="ano", columns="rede")
n1, n2 = r2[("n", "estadual_municipal")], r2[("n", "privada")]
s1, s2 = r2[("dp", "estadual_municipal")], r2[("dp", "privada")]
dp_comb = (((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2)) ** 0.5
d_rede_ind = (r2[("media", "privada")] - r2[("media", "estadual_municipal")]) / dp_comb
print(d_rede_ind.round(2).to_string())

O *d* é o mesmo, pelo menos até a segunda casa decimal. Isso confirma que a `fct_individual` com escola preenchida e a `fct_agregada` são a mesma população em 2020–2023, como a D002 já indicava. Mas isso resolve só metade do problema, porque as populações dos dois *d* continuam diferentes:

- o *d* da rede: concluintes com escola informada;
- o *d* da renda: todos os concluintes com questionário, com ou sem escola informada.

Para encerrar a questão, vamos calcular o *d* da renda só entre quem tem escola informada, a mesma população do *d* da rede:

In [ ]:
import pandas as pd

comparacao = pd.DataFrame({
    "d_renda_extremos": d,
    "d_rede_agg": d_rede,
    "d_rede_individual": d_rede_ind,
}).round(2)

In [ ]:
renda_esc = con.sql("""
    SELECT ano, faixa_renda_per_capita,
        AVG(nota_cn) AS media, STDDEV_SAMP(nota_cn) AS dp, COUNT(*) AS n
    FROM fct_individual
    WHERE tem_nota_cn AND tem_questionario AND tem_escola
    AND faixa_renda_per_capita IN ('ate_meio_sm', 'acima_2_sm')
    GROUP BY ALL
""").df().pivot(index="ano", columns="faixa_renda_per_capita")

n1, n2 = renda_esc[("n", "ate_meio_sm")], renda_esc[("n", "acima_2_sm")]
s1, s2 = renda_esc[("dp", "ate_meio_sm")], renda_esc[("dp", "acima_2_sm")]
dp_comb = (((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2)) ** 0.5
d_renda_esc = (renda_esc[("media", "acima_2_sm")] - renda_esc[("media", "ate_meio_sm")]) / dp_comb

comparacao["d_renda_com_escola"] = d_renda_esc.round(2)
print(comparacao.to_string())

| Ano | *d* da renda (com escola) | *d* da rede | Diferença |
|------|-----------------|-------|-----------|
|2020|1,35|1,18|+0,17|
|2021|1,29|1,16|+0,13|
|2022|1,16|1,01|+0,15|
|2023|1,26|1,09|+0,17|

Na mesma população, a diferença por renda continua maior que a diferença por rede em todos os anos, mas por uma margem bem menor do que antes (de 0,13 a 0,17, contra 0,2 a 0,3).

O *d* da renda cai cerca de 0,1 quando se excluem os concluintes sem escola informada. Ou seja, incluir esse grupo aumenta a diferença entre os extremos de renda. Isso indica que os concluintes sem escola informada não são um grupo neutro: eles parecem ampliar a desigualdade observada. É mais uma evidência para a ressalva da D002 de que esse grupo provavelmente não é aleatório.

Há ainda duas limitações. O *d* da renda compara os extremos de quatro faixas (deixando as duas do meio de fora), enquanto o *d* da rede compara duas redes inteiras, o que tende a favorecer a renda. Além disso, renda e rede se sobrepõem, e a separação dos seus efeitos fica para a Q3.